# PRIDE LFQ group representative report

Keep this notebook and `pride_lfq_functions.py` in the same folder. The notebook contains only inputs and calls to the functions file.


In [ ]:
# CELL 1 — dataset inputs, download, and LFQ names

from d_pride_lfq_functions import analyze_and_show, download_and_show, suggest_groups_and_show

ACCESSION = "PXD062798"

DATASET = {
    "accession": ACCESSION,
    "tissue_or_cell_culture": "heart_valve_tissue",
    "disease_type": "Heart_valve_disease_model",
    "features_list_file": "/Users/mehman/Projects/PoC_data_processing/Orthology/commons/hyperglycemia_features.txt",
    "report_file": f"{ACCESSION}_representative_report.xlsx",
    "download_dir": "PRIDE_downloads",
    "force_redownload": False,
    "timeout": 300,
    "max_full_zip_download_gb": 5.0,
}
downloaded_tables, lfq_catalog = download_and_show(DATASET)


Found 1 LFQ-containing proteinGroups table(s).


,table_id,lfq_count,gene_sources,local_file
0,table_1,6,gene column: Gene names + FASTA GN= + UniProt ...,PRIDE_downloads/PXD062798/proteinGroups_files/...


LFQ columns (6 total):


,table_id,LFQ_column_name
0,table_1,LFQ intensity CTRL_1
1,table_1,LFQ intensity CTRL_2
2,table_1,LFQ intensity CTRL_3
3,table_1,LFQ intensity RHD_1
4,table_1,LFQ intensity RHD_2
5,table_1,LFQ intensity RHD_3


In [17]:
# CELL 2 — suggest LFQ groups and show the result

GROUPS, GROUP_TABLES, grouping_preview = suggest_groups_and_show(lfq_catalog)


,suggested_group,n_LFQ_columns,detection_rule,samples
0,CTRL,3,numeric suffix,CTRL_1; CTRL_2; CTRL_3
1,RHD,3,numeric suffix,RHD_1; RHD_2; RHD_3


Suggested 2 group(s) from 6 LFQ columns.


In [18]:
# CELL 3 — review groups and analysis settings

# Optional manual corrections after reviewing the Cell 2 preview:
# GROUPS["better_name"] = GROUPS.pop("old_suggested_name")
# GROUP_TABLES["better_name"] = GROUP_TABLES.pop("old_suggested_name")
# del GROUPS["group_to_exclude"]
# GROUP_TABLES.pop("group_to_exclude", None)
MIN_GROUP_SIZE = 3

GROUPS = {
    group: columns
    for group, columns in GROUPS.items()
    if len(columns) >= MIN_GROUP_SIZE
}

GROUP_TABLES = {
    group: table
    for group, table in GROUP_TABLES.items()
    if group in GROUPS
}

grouping_preview = grouping_preview[
    grouping_preview["n_LFQ_columns"] >= MIN_GROUP_SIZE
].reset_index(drop=True)

display(grouping_preview)

SETTINGS = {
    "group_tables": GROUP_TABLES,
    "presence_threshold": 0.10,          # strict Presence > 0.10, using all LFQ columns
    "group_aggregation": "mean",
    "duplicate_gene_aggregation": "mean",
    "treat_zero_as_missing": True,
    "allow_column_reuse": False,
}


,suggested_group,n_LFQ_columns,detection_rule,samples
0,CTRL,3,numeric suffix,CTRL_1; CTRL_2; CTRL_3
1,RHD,3,numeric suffix,RHD_1; RHD_2; RHD_3


In [21]:
# CELL 4 — run and show the results

result = analyze_and_show(DATASET, GROUPS, SETTINGS, downloaded_tables)


ValueError: Output-name components cannot be empty

## Output behavior

- The first report column is `Gene`, in the same order as `features_list`.
- Representative columns use `accession_group_tissue_or_cell_culture_disease`.
- Genes that match annotations but do not satisfy strict whole-table `Presence > 0.10` remain blank in the representative columns.
- Rerunning with another accession or new groups retains earlier report columns.
- If the feature list changes, use a different report filename.
- `result["feature_status"]` contains the gene-level annotation and eligibility flags when detailed checking is needed.
